# Як модель навчається

## Що ми вже знаємо

* **Модель** — це формула, всередині якої є числа-налаштування. Їх називають **вагами**.
* Наприклад, модель «витрата пального = $w$ × маса машини». Тут $w$ — вага. Якщо $w$ вдале, модель вгадує добре. Якщо ні — погано.
* **Помилку** моделі ми вміємо рахувати: беремо прогноз, беремо правильну відповідь і дивимось, наскільки вони різні.

## Чого ми ще не знаємо

Ми багато разів казали «модель навчилася». Але що це означає? Хто підбирає $w$? Як?

Сьогодні ми самі, крок за кроком, придумаємо, як комп'ютер знаходить хороші ваги. Наприкінці виявиться, що ми придумали той самий спосіб, яким тренують справжні нейромережі, зокрема ChatGPT.

## Як буде йти заняття

Історія одна, і кожен наступний крок виростає з проблеми, на яку ми натрапили в попередньому:

1. Беремо одну машину і дивимось, як помилка залежить від $w$.
2. Розуміємо, що перебрати всі варіанти неможливо, і треба шукати розумніше.
3. Придумуємо, як зрозуміти, куди змінювати $w$ і наскільки.
4. Складаємо з цього правило, яке можна повторювати — це і буде «навчання».
5. Переходимо до справжніх машин.
6. Думаємо, що робити, коли даних мільйони.
7. Думаємо, що робити, коли ваг багато і вони стоять одна за одною, як у нейромережі.

## Як працювати з ноутбуком

* Жирним виділено **питання**. Спершу подумайте самі, можна вголос і всі разом.
* Під деякими питаннями захована **Підказка** (натисніть, щоб розгорнути). Відкривайте її, тільки якщо зовсім застрягли.
* Під питанням є поле **«Ваша відповідь»**: двічі клацніть по ньому і запишіть свій варіант. Відповіді ми обговоримо разом.
* Після кожного кроку є **Проміжний підсумок**: що ми зробили, навіщо, і яка проблема веде нас далі. Якщо загубились, читайте підсумки.
* Клітинки з кодом бувають двох видів:
  * **«Рахуємо»** — там тільки арифметика і `print`. Міняйте числа і запускайте (Shift+Enter).
  * **«Для демонстрації»** — код для малювання графіків. Читати його не треба, просто запустіть і дивіться на картинку.

In [ ]:
# Для демонстрації: підключаємо бібліотеку для графіків
import csv
import random
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

---
## Крок 1. Одна машина

Почнемо з найпростішого, що можна уявити: **одна машина**. Вона важить **1 тонну** і витрачає **8 літрів на 100 км**. (Приблизно така була Toyota Corolla 1975 року: 0.98 т і 8.1 л. Числа округлили, щоб рахувати в голові.)

Позначення, які будемо використовувати весь час:

* $x$ — те, що ми знаємо про машину. Тут це маса: $x = 1$.
* $y$ — правильна відповідь, яку модель має вгадати. Тут це витрата: $y = 8$.
* $w$ — вага моделі, яку ми шукаємо.
* прогноз моделі $= w \cdot x$.

Спробуємо $w = 5$. Прогноз $= 5 \cdot 1 = 5$ літрів. А насправді 8. Помилились на $5 - 8 = -3$ літри.

### Як виміряти, наскільки погано ми помилились?

Різниця «прогноз мінус правда» буває і додатною (переоцінили), і від'ємною (недооцінили). Нас же цікавить не напрямок, а **наскільки погано**. Помилка на $-3$ так само погана, як і на $+3$.

Щоб позбутися знака, різницю **підносять до квадрата**: $(-3)^2 = 9$ і $(+3)^2 = 9$. Ви вже бачили це в MSE. Отже, помилка моделі:

$$\text{Помилка}(w) = (\text{прогноз} - \text{правда})^2 = (w \cdot 1 - 8)^2 = (w - 8)^2$$

**Завдання.** Порахуйте помилку для різних $w$ (у голові або в клітинці нижче) і заповніть таблицю:

| $w$ | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 |
|---|---|---|---|---|---|---|---|---|---|
| прогноз | 4 | 5 | | | | | | | |
| помилка $(w-8)^2$ | 16 | 9 | | | | | | | |

**Питання 1.** Яке $w$ найкраще? Що відбувається з помилкою, коли віддаляємось від нього в будь-який бік?

In [ ]:
# Рахуємо: змінюйте w і запускайте ще раз
w = 5
x = 1      # маса, тонн
y = 8      # справжня витрата, л/100 км

prediction = w * x
error = (prediction - y) ** 2       # ** 2 означає "в квадраті"
print("w =", w, "  прогноз =", prediction, "  помилка =", error)

**Ваша відповідь:** _двічі клацніть по цій клітинці, щоб написати_

In [ ]:
# Для демонстрації: графік помилки для однієї машини
ws = [i / 10 for i in range(30, 131)]
plt.plot(ws, [(w - 8) ** 2 for w in ws])
plt.scatter(range(4, 13), [(w - 8) ** 2 for w in range(4, 13)], color="red", zorder=3, label="точки з таблиці")
plt.xlabel("w (вага моделі)")
plt.ylabel("помилка")
plt.title("Помилка моделі для однієї машини")
plt.legend()
plt.show()

Цей графік дуже важливий, тому скажемо це ще раз простими словами:

* кожна точка по горизонталі — це **одна версія моделі** (одне значення $w$);
* висота точки — **наскільки ця версія погана**;
* **навчити модель** = знайти, де ця яма найглибша.

Далі часто будемо казати «яма», «схил», «дно». Це все про цей графік.

### Проміжний підсумок

* **Що зробили:** Взяли одну машину і модель «прогноз = w × маса». Порахували помилку для різних w і побачили, що графік помилки — це яма. Найкраще w = 8, там дно.
* **Навіщо:** Тепер у нас є точне формулювання, що означає «навчити модель»: **знайти w, при якому помилка найменша, тобто дно ями**.
* **Що далі:** Ми знайшли дно перебором: спробували 9 значень. Питання — чи можна так робити завжди?

---
## Крок 2. Чому не можна просто перебрати всі варіанти?

Ми щойно знайшли найкраще $w$ дуже просто: перебрали 9 значень і подивились, де помилка найменша. Чому б комп'ютеру не робити так само?

**Питання 2.** У справжніх моделях ваг не одна, а багато. Нехай для кожної ваги ми хочемо перевірити 100 різних значень. Скільки всього варіантів доведеться перевірити, якщо ваг:

* 1?
* 2?
* 10?


<details>
<summary><b>Підказка</b></summary>

Для двох ваг: кожне з 100 значень першої ваги треба спробувати з кожним із 100 значень другої. Як у кодовому замку з двома коліщатками по 100 положень.

</details>

**Ваша відповідь:** _двічі клацніть по цій клітинці, щоб написати_

### Гора в тумані

Уявіть, що ви стоїте на схилі гори. Густий туман, видно тільки землю під ногами. Треба спуститися в найнижчу точку. Карти немає.

Це рівно ситуація комп'ютера. Весь графік помилки він не бачить, бо для цього треба перебрати всі варіанти. Але в тій точці, де він зараз стоїть (при поточних вагах), він може порахувати помилку. І може порахувати її **трохи поруч**.

**Питання 3.** Як би ви спускалися з гори в тумані? Опишіть покроково, так, щоб це міг виконати робот.


**Ваша відповідь:** _двічі клацніть по цій клітинці, щоб написати_

### Проміжний підсумок

* **Що зробили:** Порахували, що перебір неможливий, коли ваг багато: вже на 10 вагах це тисячі років.
* **Навіщо:** Звідси головне обмеження: комп'ютер як у тумані. Він знає помилку тільки там, де стоїть зараз, і може порахувати її трохи поруч. План: дивитись поруч і крокувати вниз.
* **Що далі:** Треба зрозуміти, як, дивлячись тільки поруч, визначити, куди крокувати.

---
## Крок 3. Куди йти?

Граємо чесно: **ми не знаємо, що найкраще $w = 8$.** Ми стоїмо в $w = 12$, і більше нічого не бачимо.

«Помацати ногою» — означає порахувати помилку трохи ліворуч і трохи праворуч, наприклад на відстані $0.1$.

**Завдання.** Порахуйте помилку в $w = 11.9$, $w = 12$ і $w = 12.1$.

**Питання 4.** Куди змінювати $w$: зменшувати чи збільшувати?

In [ ]:
# Рахуємо
error_left  = (11.9 - 8) ** 2
error_here  = (12.0 - 8) ** 2
error_right = (12.1 - 8) ** 2
print("ліворуч:", error_left, "  тут:", error_here, "  праворуч:", error_right)

**Ваша відповідь:** _двічі клацніть по цій клітинці, щоб написати_

### Проміжний підсумок

* **Що зробили:** Порахували помилку трохи ліворуч і трохи праворуч від w = 12 і побачили, що ліворуч нижче.
* **Навіщо:** Так ми дізнались **напрямок**, не знаючи, де дно, і не малюючи графіка.
* **Що далі:** Напрямок є, але незрозуміло, **наскільки** змінювати w: на 0.1, на 1, на 10?

---
## Крок 4. Наскільки круто?

Напрямок знаємо. Але **наскільки** змінювати $w$? На $0.1$? На $1$? На $10$?

**Питання 5.** Згадайте гору. Ви стоїте на крутому обриві — дно, найімовірніше, близько чи далеко? А якщо під ногами майже рівно?


**Ваша відповідь:** _двічі клацніть по цій клітинці, щоб написати_

### Як виміряти крутизну числом

Згадайте знак на дорозі «ухил 10%». Він означає: на кожні 100 м уперед дорога опускається на 10 м, тобто на кожен метр уперед — на $0.1$ м. **Крутизна = наскільки змінилась висота, поділене на те, наскільки ми пройшли вперед.**

У кроці 3 ми відступили від $w = 12$ праворуч на $0.1$ і побачили, що помилка зросла з $16$ до $16.81$, тобто на $0.81$.

**Питання 6.** Чому не можна просто сказати «крутизна = 0.81»? Що буде, якщо відступити не на $0.1$, а на $1$?


**Ваша відповідь:** _двічі клацніть по цій клітинці, щоб написати_

Запишемо це формулою. Нехай ми стоїмо в $w$ і відступаємо на маленьку відстань $h$:

$$\text{крутизна} \approx \frac{\text{Помилка}(w + h) - \text{Помилка}(w)}{h} = \frac{\text{на скільки змінилась помилка}}{\text{на скільки ми змістились}}$$

**Завдання.** Порахуйте крутизну в $w = 12$ для різних $h$:

| $h$ | 1 | 0.1 | 0.01 | 0.001 |
|---|---|---|---|---|
| крутизна | 9 | 8.1 | | |

In [ ]:
# Рахуємо: міняйте h
w = 12
h = 0.1
slope = ((w + h - 8) ** 2 - (w - 8) ** 2) / h
print("h =", h, "  крутизна ≈", round(slope, 6))    # round(число, 6) округлює до 6 знаків після коми

**Питання 7.** До якого числа наближається крутизна, коли $h$ зменшується? Чому з великим $h$ виходить інакше?


<details>
<summary><b>Підказка</b></summary>

Подивіться на графік нижче. Пряма через дві точки ями — це «середній схил» між ними. Яма вигнута: що далі другу точку взяли, то більше в цей «середній схил» потрапляє крутіших ділянок.

</details>

**Ваша відповідь:** _двічі клацніть по цій клітинці, щоб написати_

In [ ]:
# Для демонстрації: пряма через дві точки. Що менший h, то ближче вона до схилу в самій точці
ws = [i / 10 for i in range(60, 141)]
plt.plot(ws, [(w - 8) ** 2 for w in ws], color="black", label="помилка")
for h, c in [(2, "tab:red"), (1, "tab:orange"), (0.3, "tab:green")]:
    k = ((12 + h - 8) ** 2 - 16) / h
    plt.plot(ws, [16 + k * (w - 12) for w in ws], color=c, label=f"h = {h}: крутизна {k:.1f}")
    plt.scatter([12 + h], [(12 + h - 8) ** 2], color=c, zorder=3)
plt.plot(ws, [16 + 8 * (w - 12) for w in ws], "--", color="tab:blue", label="h дуже малий: крутизна 8")
plt.scatter([12], [16], color="black", zorder=4)
plt.ylim(-5, 45)
plt.xlabel("w"); plt.ylabel("помилка")
plt.legend()
plt.show()

### Шукаємо закономірність

Щоразу рахувати крутизну через $h$ незручно. Може, є простіший спосіб? Порахуємо «справжню» крутизну (з дуже малим $h = 0.0001$) в різних точках і пошукаємо закономірність.

| $w$ | 12 | 11 | 10 | 9 | 8 | 7 | 6 | 4 |
|---|---|---|---|---|---|---|---|---|
| крутизна | 8 | | | | | | | |
| $w - 8$ (відстань до дна) | 4 | 3 | 2 | 1 | 0 | −1 | −2 | −4 |

In [ ]:
# Рахуємо: міняйте w
w = 12
h = 0.0001
slope = ((w + h - 8) ** 2 - (w - 8) ** 2) / h
print("w =", w, "  крутизна ≈", round(slope, 3))

**Питання 8.** Порівняйте рядок «крутизна» з рядком «$w - 8$». Як одне отримати з іншого?


**Ваша відповідь:** _двічі клацніть по цій клітинці, щоб написати_

<details>
<summary><b>Для допитливих:</b> чому саме 2·(w − 8)? (можна пропустити)</summary>

Позначимо відстань до дна $a = w - 8$. Помилка — це $a^2$, тобто площа квадрата зі стороною $a$.

Збільшимо сторону на крихітне $h$. Новий квадрат $(a+h)^2$ складається зі старого квадрата $a^2$, двох довгих тонких смужок $a \times h$ і крихітного квадратика $h \times h$:

$$(a + h)^2 = a^2 + 2ah + h^2$$

Отже, площа зросла на $2ah + h^2$. Ділимо на $h$: крутизна $= 2a + h$.

Тепер видно все з таблиці: з $h = 0.1$ маємо $8 + 0.1 = 8.1$, з $h = 0.01$ — $8.01$. Коли $h$ крихітний, доданок $h$ зникає, і залишається $2a = 2(w - 8)$.

</details>

### Що означає знак крутизни

**Питання 9.** Порахуйте крутизну в $w = 12$, $w = 8$ і $w = 4$. Коли вона додатна, коли від'ємна, коли нуль? Що кожен випадок каже про те, куди йти?


**Ваша відповідь:** _двічі клацніть по цій клітинці, щоб написати_

### Назва

Те, що ви зараз придумали — «наскільки змінюється результат, якщо трохи змінити $w$, у розрахунку на одиницю зміни» — у математиці називається **похідна**.

У книжках і статтях ви побачите її так: $\dfrac{dL}{dw}$ (читається «де ел по де дубль-ве»). Буква $d$ — від слова «різниця» (difference), $L$ — помилка (від англійського loss, «втрата»). Тобто запис буквально означає: «маленька зміна помилки, поділена на маленьку зміну $w$». Рівно те, що ми рахували.

Звідки це взялось. У 1630-х роках француз П'єр Ферма шукав, де функції набувають найменших і найбільших значень. Він помітив те саме, що й ми: на самому дні ями крутизна нульова. Через кілька десятиліть Ісаак Ньютон в Англії і Готфрід Лейбніц у Німеччині, незалежно один від одного, перетворили цю ідею на цілу науку. Стаття Лейбніца 1684 року, в якій вперше з'явилось позначення $d$, так і називалась: «Новий метод для максимумів і мінімумів». Тобто похідну від самого початку придумали для того самого, що робимо ми: **шукати дно**.

### Проміжний підсумок

* **Що зробили:** Придумали число «крутизна» = (зміна помилки) / (зміна w), коли зміна дуже мала. Для помилки (w − 8)² знайшли закономірність: крутизна = 2 · (w − 8). Звідси **правило квадрата: крутизна квадрата = 2 · те, що в квадраті**. Це число називається **похідна**.
* **Навіщо:** Одне число відповідає на обидва питання: **знак** каже, куди йти (завжди проти знака), **величина** — наскільки круто, тобто чи далеко ще до дна.
* **Що далі:** Перетворити це знання на конкретне правило: як із крутизни отримати нове w.

---
## Крок 5. Правило кроку

Складемо з усього цього правило, як змінювати $w$. Ми хочемо, щоб:

1. якщо крутизна додатна, $w$ **зменшувалось**;
2. якщо крутизна від'ємна, $w$ **збільшувалось**;
3. що крутіше, то більший крок; на дні (крутизна 0) — стоїмо на місці.

**Питання 10.** Яка найпростіша формула виконує всі три вимоги?

$$w_{\text{нове}} = w_{\text{старе}} \ \boxed{\ ?\ }$$


<details>
<summary><b>Підказка</b></summary>

Спробуйте відняти від $w$ саму крутизну. Перевірте всі три вимоги на прикладах: крутизна $+8$, $-8$, $0$.

</details>

**Ваша відповідь:** _двічі клацніть по цій клітинці, щоб написати_

**Питання 11.** Спробуйте це правило. Ми в $w = 12$, крутизна $8$. Куди потрапимо? Яка там помилка? Нам це подобається?


**Ваша відповідь:** _двічі клацніть по цій клітинці, щоб написати_

Крок завеликий — значить, треба його зменшити. Помножимо крутизну на маленьке число, наприклад $0.1$:

$$\boxed{w_{\text{нове}} = w_{\text{старе}} - \text{lr} \cdot \text{крутизна}}$$

Це маленьке число називають **lr**, від англійського *learning rate*, «швидкість навчання». Ми його не вираховуємо, а **вибираємо самі**. Як саме вибирати — зараз побачимо.

А сама формула — це і є спосіб, яким навчаються майже всі сучасні моделі. Називається він **градієнтний спуск** (gradient descent). «Спуск» — бо спускаємось на дно ями. Що таке «градієнт», скажемо пізніше, коли в нас буде кілька ваг.

Першим цей метод описав французький математик Огюстен Коші 1847 року. Він розв'язував громіздкі рівняння для астрономії і запропонував рухатися маленькими кроками туди, де помилка менша.

### Проміжний підсумок

* **Що зробили:** Склали правило: **w нове = w старе − lr · крутизна**. Мінус — бо йдемо проти крутизни. Множник lr — бо без нього крок завеликий і ми перестрибуємо дно.
* **Навіщо:** Це правило можна повторювати раз за разом, і кожен раз ми трохи ближче до дна. Це і є **градієнтний спуск**, спосіб навчання майже всіх сучасних моделей.
* **Що далі:** Перевірити, чи правило справді працює, і розібратись, як вибирати lr.

---
## Крок 6. Навчаємо! І підбираємо розмір кроку

Стартуємо з $w = 12$, $\text{lr} = 0.1$.

Нижче дві клітинки. Першу запустіть **один раз**: вона ставить нас на старт. Другу запускайте **багато разів поспіль**: кожен запуск — один крок униз. Так ми виконуємо алгоритм руками.

| крок | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| $w$ | 12 | | | | | |
| помилка | 16 | | | | | |

In [ ]:
# Рахуємо: СТАРТ (запустіть один раз)
w = 12
lr = 0.1

In [ ]:
# Рахуємо: ОДИН КРОК (запускайте раз за разом)
slope = 2 * (w - 8)          # крутизна в точці, де ми стоїмо
w = w - lr * slope           # правило кроку
print("w =", round(w, 4), "  помилка =", round((w - 8) ** 2, 4))

**Ваша відповідь:** _двічі клацніть по цій клітинці, щоб написати_

**Питання 12.** Поверніться на старт і повторіть для $\text{lr} = 0.01$, $\text{lr} = 1$, $\text{lr} = 1.1$. Зробіть по 4-5 кроків. Опишіть кожен випадок одним-двома словами.


**Ваша відповідь:** _двічі клацніть по цій клітинці, щоб написати_

In [ ]:
# Для демонстрації: 8 кроків з різними lr
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5), sharey=True)
ws = [i / 10 for i in range(-20, 181)]
for ax, lr in zip(axes, [0.01, 0.1, 1.0, 1.1]):
    path = [12.0]
    for _ in range(8):
        path.append(path[-1] - lr * 2 * (path[-1] - 8))
    ax.plot(ws, [(w - 8) ** 2 for w in ws], color="lightgray")
    ax.plot(path, [(w - 8) ** 2 for w in path], "o-", color="tab:red", markersize=4)
    ax.set_title(f"lr = {lr}")
    ax.set_ylim(-3, 100)
    ax.set_xlabel("w")
plt.show()

Отже, про learning rate треба пам'ятати три речі:

* **замалий** — навчання йде дуже довго;
* **завеликий** — модель стрибає туди-сюди або взагалі «вилітає»;
* правильного значення на всі випадки **немає**, його підбирають експериментами.

Коли колись побачите в коді навчання `lr=0.001` чи `learning_rate=0.0003`, це воно і є.

### Проміжний підсумок

* **Що зробили:** Повторювали крок за кроком і дійшли до дна w = 8, ніде не використовуючи знання, де воно. Побачили, що кроки самі зменшуються біля дна. Побачили, що lr замалий — повзе, завеликий — стрибає або вилітає.
* **Навіщо:** Тепер ми маємо робочий спосіб навчання і розуміємо його головне налаштування — learning rate.
* **Що далі:** Досі була одна вигадана машина з масою рівно 1 т. Чи працює все це на справжніх даних?

---
## Крок 7. Справжні машини

Досі в нас була одна вигадана машина. Тепер беремо справжні дані: датасет **Auto MPG** з репозиторію UCI. Це 398 справжніх автомобілів 1970–1982 років. Його дуже часто використовують, щоб вчитися ML. В оригіналі маса у фунтах, а витрата в «милях на галон», тому ми перевели в тонни і літри на 100 км. (У 6 машин бракує частини даних, їх викинули, лишилось 392.)

Подивимось, чи взагалі є зв'язок між масою і витратою:

In [ ]:
# Для демонстрації: завантажуємо всі 392 машини з файлу auto_mpg.csv
# (файл має лежати в тій самій папці, що й ноутбук)
X_all = []   # маси, т
Y_all = []   # витрати, л/100 км
with open("auto_mpg.csv") as f:
    for row in csv.DictReader(f):
        X_all.append(float(row["mass_t"]))
        Y_all.append(float(row["litres_per_100km"]))
print("Машин:", len(X_all))

plt.scatter(X_all, Y_all, s=10, alpha=0.5)
plt.xlabel("маса, т"); plt.ylabel("витрата, л/100 км")
plt.title("Auto MPG: кожна точка — одна машина")
plt.show()

Зв'язок явно є: що важча машина, то більше вона витрачає. Модель та сама: прогноз $= w \cdot x$, тобто «скільки літрів додає кожна тонна маси».

Для ручних обчислень візьмемо три машини (числа округлили):

| Машина | Маса $x$, т | Витрата $y$, л/100 км |
|---|---|---|
| Volkswagen 1131 (1970) | 0.83 | 9.0 |
| Ford Maverick (1970) | 1.17 | 11.2 |
| Chevrolet Impala (1970) | 1.97 | 16.8 |

**Питання 13.** Як тепер порахувати помилку моделі? Машин три, а число помилки нам потрібне одне.


**Ваша відповідь:** _двічі клацніть по цій клітинці, щоб написати_

In [ ]:
# Рахуємо: помилка на трьох машинах. Спробуйте w = 5, 8, 9, 10, 12
w = 8
e1 = (w * 0.83 - 9.0) ** 2     # Volkswagen
e2 = (w * 1.17 - 11.2) ** 2    # Ford
e3 = (w * 1.97 - 16.8) ** 2    # Chevrolet
print("w =", w, "  помилка =", round((e1 + e2 + e3) / 3, 3))

In [ ]:
# Для демонстрації: графік помилки на трьох машинах
X3 = [0.83, 1.17, 1.97]
Y3 = [9.0, 11.2, 16.8]
ws = [i / 20 for i in range(80, 260)]
plt.plot(ws, [sum((w * x - y) ** 2 for x, y in zip(X3, Y3)) / 3 for w in ws])
plt.xlabel("w (вага моделі)"); plt.ylabel("середня помилка")
plt.title("Три справжні машини: знову яма")
plt.show()

Знову яма. Значить, можна спускатися так само: $w_{\text{нове}} = w_{\text{старе}} - \text{lr} \cdot \text{крутизна}$.

Лишилось зрозуміти, як рахувати крутизну. Для однієї вигаданої машини ми вивели $2 \cdot (w - 8)$. Але там маса була рівно $1$. Що змінюється, коли маса інша?

### Крутизна для однієї машини

Візьмемо Volkswagen: $x = 0.83$, $y = 9.0$. Помилка $= (w \cdot 0.83 - 9.0)^2$. Розберемо, що відбувається, коли ми трохи змінюємо $w$. Тут **два етапи**.

**Етап 1. Як $w$ змінює прогноз?** Прогноз $= w \cdot 0.83$.

**Питання 14.** Якщо $w$ збільшити на $0.1$ (з 8 до 8.1), на скільки зміниться прогноз? А якщо на $1$?


**Ваша відповідь:** _двічі клацніть по цій клітинці, щоб написати_

**Етап 2. Як прогноз змінює помилку?** Помилка $= (\text{прогноз} - 9.0)^2$. Тут у нас знову квадрат, а для квадрата ми вже знаємо правило з кроку 4: **крутизна $= 2 \cdot$ те, що в квадраті**, тобто $2 \cdot (\text{прогноз} - 9.0)$.

При $w = 8$: прогноз $6.64$, отже крутизна по прогнозу $= 2 \cdot (6.64 - 9.0) = -4.72$.

**Питання 15.** Складіть два етапи разом. Змінюємо $w$ на крихітну величину. Прогноз змінюється в $0.83$ раза сильніше. Помилка змінюється в $-4.72$ раза сильніше, ніж прогноз. У скільки разів помилка змінюється сильніше за $w$?


<details>
<summary><b>Підказка</b></summary>

Аналогія з грошима. 1 долар = 41 гривня, 1 євро = 1.1 долара. Скільки гривень в одному євро? Що ви робите з двома курсами?

</details>

**Ваша відповідь:** _двічі клацніть по цій клітинці, щоб написати_

Не віримо на слово, перевіряємо старим способом, через маленьке $h$:

In [ ]:
# Рахуємо: крутизна для Volkswagen двома способами
w = 8
x = 0.83
y = 9.0

# Спосіб 1: формула "2 * (прогноз - y) * x"
slope_formula = 2 * (w * x - y) * x

# Спосіб 2: "в лоб", як у кроці 4
h = 0.0001
slope_direct = ((( w + h) * x - y) ** 2 - (w * x - y) ** 2) / h

print("за формулою:", round(slope_formula, 4), "   в лоб:", round(slope_direct, 4))

Збігається.

У цій формулі є глибокий сенс, який варто промовити вголос. Крутизна $= 2 \cdot \text{помилка прогнозу} \cdot \text{вхід}$:

* **помилка прогнозу**: якщо прогноз ідеальний, вона $0$, і $w$ чіпати не треба. Що більше помиляємось, то сильніше тягнемо $w$;
* **вхід $x$**: якщо машина важила б $0$ тонн, $w$ ніяк не вплинув би на прогноз, і «звинувачувати» його в помилці було б нечесно.

Кожна вага в нейромережі змінюється за цим самим принципом: **наскільки помилились × наскільки ця вага була причетна**.

### Крутизна для трьох машин

Помилка — це середнє трьох помилок. Отже, і крутизна — **середнє трьох крутизн**. (Якщо кожна з трьох помилок зростає, скажімо, на 3, на 6 і на 9, то їхнє середнє зростає на середнє $(3+6+9)/3 = 6$.)

**Завдання.** Навчіть модель на трьох машинах. Старт $w = 5$, $\text{lr} = 0.1$, зробіть 5 кроків.

| крок | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| $w$ | 5 | | | | | |
| помилка | 33.48 | | | | | |

In [ ]:
# Рахуємо: СТАРТ (один раз)
w = 5
lr = 0.1

In [ ]:
# Рахуємо: ОДИН КРОК на трьох машинах (запускайте раз за разом)
slope_vw    = 2 * (w * 0.83 - 9.0)  * 0.83
slope_ford  = 2 * (w * 1.17 - 11.2) * 1.17
slope_chevy = 2 * (w * 1.97 - 16.8) * 1.97
slope = (slope_vw + slope_ford + slope_chevy) / 3    # середня крутизна

w = w - lr * slope

error = ((w * 0.83 - 9.0) ** 2 + (w * 1.17 - 11.2) ** 2 + (w * 1.97 - 16.8) ** 2) / 3
print("w =", round(w, 3), "  помилка =", round(error, 3))

**Питання 16.** До якого $w$ ви прийшли? Що це число означає для машин?


**Ваша відповідь:** _двічі клацніть по цій клітинці, щоб написати_

### Проміжний підсумок

* **Що зробили:** Взяли три справжні машини. Помилка — середнє помилок (MSE), і графік знову яма. Вивели крутизну для однієї машини: **2 · (прогноз − y) · x**, як добуток двох етапів: як w впливає на прогноз (у x разів) і як прогноз впливає на помилку (правило квадрата). Для кількох машин — середнє. Навчили модель: w ≈ 9 літрів на кожну тонну.
* **Навіщо:** Тепер градієнтний спуск працює на справжніх даних з будь-якою масою машини, а не тільки на вигаданому прикладі.
* **Що далі:** На кожен крок ми рахували крутизну для **кожної** машини. А якщо машин мільйони?

---
## Крок 8. А якщо даних дуже багато?

Щоб зробити **один** крок, ми рахували крутизну **для кожної машини** й усереднювали.

* 3 машини — 3 обчислення на крок. Легко.
* 392 машини — 392 обчислення на крок. Комп'ютеру теж нормально.
* Мовні моделі вчать на трильйонах слів, і кроків треба сотні тисяч.

**Питання 17.** Перед кожним маленьким кроком перераховувати весь інтернет? Що запропонуєте?


<details>
<summary><b>Підказка</b></summary>

Як соціологи дізнаються, за кого проголосує країна? Вони ж не питають кожну з мільйонів людей.

</details>

**Ваша відповідь:** _двічі клацніть по цій клітинці, щоб написати_

Перевіримо, наскільки неточна оцінка з однієї машини. Порахуйте крутизну для кожної з трьох машин **окремо** в двох точках: $w = 8$ (далеко від дна $\approx 9.04$) і $w = 10$ (трохи проскочили дно).

| | Volkswagen | Ford | Chevrolet |
|---|---|---|---|
| $w = 8$ | | | |
| $w = 10$ | | | |

In [ ]:
# Рахуємо: крутизна для кожної машини окремо. Спробуйте w = 8 і w = 10
w = 8
print("Volkswagen:", round(2 * (w * 0.83 - 9.0)  * 0.83, 2))
print("Ford:      ", round(2 * (w * 1.17 - 11.2) * 1.17, 2))
print("Chevrolet: ", round(2 * (w * 1.97 - 16.8) * 1.97, 2))

**Питання 18.** Чи «згодні» машини між собою, куди рухати $w$? Коли так, а коли ні?


**Ваша відповідь:** _двічі клацніть по цій клітинці, щоб написати_

Тепер подивимось на всі 392 машини і порівняємо три способи. Модель та сама: прогноз $= w \cdot x$. На всіх машинах найкраще $w \approx 8.4$ (пунктир на графіку).

In [ ]:
# Для демонстрації: як змінюється w крок за кроком при трьох способах
def train(batch_size, steps=60, lr=0.1, seed=1):
    rnd = random.Random(seed)
    w = 2.0
    history = [w]
    for _ in range(steps):
        if batch_size is None:
            idx = range(len(X_all))                           # усі машини
        else:
            idx = rnd.sample(range(len(X_all)), batch_size)   # випадкові машини
        slope = sum(2 * (w * X_all[i] - Y_all[i]) * X_all[i] for i in idx) / len(idx)
        w = w - lr * slope
        history.append(w)
    return history

plt.figure(figsize=(9, 4.5))
plt.plot(train(None), "o-", markersize=3, label="усі 392 машини на крок (392 обчислення)")
plt.plot(train(32), "o-", markersize=3, label="32 випадкові машини на крок (32 обчислення)")
plt.plot(train(1), "o-", markersize=3, label="1 випадкова машина на крок (1 обчислення)")
plt.axhline(8.38, color="black", linestyle="--", label="найкраще w ≈ 8.4")
plt.xlabel("номер кроку"); plt.ylabel("w")
plt.legend()
plt.show()

Що видно:

* **Усі машини**: найрівніший шлях. Але кожен крок коштує 392 обчислення.
* **32 машини**: майже такий самий шлях, а крок у 12 разів дешевший.
* **1 машина**: доходить туди ж, але дорогою «тремтить», і біля дна теж. Зате крок коштує лише 1 обчислення.

Якщо часу вистачає на 392 обчислення, то з усіма машинами ми встигнемо зробити **1 крок**, а з однією випадковою машиною — **392 кроки**. Тому на практиці майже завжди беруть невеликі випадкові пачки.

### Назви

| Що ми придумали | Як це називається |
|---|---|
| усі дані на кожен крок | **batch gradient descent** |
| одна випадкова машина на крок | **SGD**, stochastic gradient descent («стохастичний» = випадковий) |
| кілька випадкових машин на крок | **mini-batch gradient descent** |

Кількість прикладів на крок називають **batch size** («розмір пачки»). Ви побачите його в будь-якому коді навчання поруч із learning rate. Через звичку mini-batch теж часто називають просто «SGD».

Ідея оцінювати напрямок за випадковими прикладами старша за нейромережі: її описали Герберт Роббінс і Саттон Монро у 1951 році.

### Проміжний підсумок

* **Що зробили:** Замість усіх машин на кожен крок беремо випадкову вибірку, як соціологічне опитування: одну машину (**SGD**) або кілька (**mini-batch**). Побачили, що далеко від дна вибірка майже завжди показує правильний напрямок, а біля дна модель трохи «тремтить».
* **Навіщо:** Кожен крок стає в сотні й тисячі разів дешевшим, і тому можна навчатися на величезних даних.
* **Що далі:** Досі в моделі була одна вага. У нейромережі ваг багато, і вони стоять одна за одною. Як порахувати крутизну для кожної з них?

---
## Крок 9. Кілька ваг одна за одною: як у нейромережі

Досі в моделі була одна вага. У нейромережі ваг багато, і вони стоять **одна за одною**: вхід множиться на вагу, результат іде далі, знову множиться на іншу вагу, і так багато разів.

Візьмемо найменшу таку «мережу» — дві ваги підряд:

$$x \ \xrightarrow{\ \times w_1\ }\ h \ \xrightarrow{\ \times w_2\ }\ \text{прогноз} \ \longrightarrow\ \text{помилка} = (\text{прогноз} - y)^2$$

Словами:

1. вхід $x$ множимо на першу вагу $w_1$ і отримуємо проміжне число $h$ («прихований шар»);
2. $h$ множимо на другу вагу $w_2$ і отримуємо прогноз;
3. рахуємо помилку, як завжди.

Візьмемо прості числа, щоб рахувати в голові:

$$x = 2,\quad y = 4,\quad w_1 = 0.5,\quad w_2 = 3$$

**Завдання 1.** Порахуйте $h$, прогноз і помилку. Це називається **прямий прохід**: ідемо від входу до помилки.

**Ваша відповідь:** _двічі клацніть по цій клітинці, щоб написати_

Тепер треба змінити **обидві** ваги. Для цього потрібна крутизна по кожній: наскільки зміниться помилка, якщо трохи змінити $w_1$? А якщо $w_2$?

Робимо так само, як у кроці 7: розбиваємо шлях на етапи і дивимось, у скільки разів кожен етап «підсилює» зміну.

**Питання 19.** Розберіть кожен етап окремо:

* а) Прогноз $= w_2 \cdot h$. Якщо **прогноз** зміниться на трішки, у скільки разів сильніше зміниться **помилка**?
* б) Якщо **$w_2$** зміниться на трішки, у скільки разів сильніше зміниться **прогноз**?
* в) Якщо **$h$** зміниться на трішки, у скільки разів сильніше зміниться **прогноз**?
* г) $h = w_1 \cdot x$. Якщо **$w_1$** зміниться на трішки, у скільки разів сильніше зміниться **$h$**?


<details>
<summary><b>Підказка</b></summary>

Кожен етап — це або «помножити на число», або «піднести до квадрата». Для «помножити на число» відповідь дає Питання 14: зміна множиться на те число, на яке множимо. Для квадрата — правило «$2 \cdot$ те, що в квадраті».

Можна перевірити на числах: змініть $w_2$ з $3$ на $3.1$ і подивіться, що стало з прогнозом.

</details>

**Ваша відповідь:** _двічі клацніть по цій клітинці, щоб написати_

**Питання 20.** Тепер складіть ланцюжки, так само як із курсами валют:

* для $w_2$: $w_2 \to$ прогноз $\to$ помилка;
* для $w_1$: $w_1 \to h \to$ прогноз $\to$ помилка.

Яка крутизна по $w_2$? По $w_1$?


**Ваша відповідь:** _двічі клацніть по цій клітинці, щоб написати_

### Чому обчислюють «назад»

Подивіться, з чого починаються обидва ланцюжки: з етапу (а) «як прогноз впливає на помилку», тобто з $-2$. Далі:

| що рахуємо | як | результат |
|---|---|---|
| як прогноз впливає на помилку | $2 \cdot (3 - 4)$ | $-2$ |
| крутизна по $w_2$ | $-2 \cdot h = -2 \cdot 1$ | $-2$ |
| як $h$ впливає на помилку | $-2 \cdot w_2 = -2 \cdot 3$ | $-6$ |
| крутизна по $w_1$ | $-6 \cdot x = -6 \cdot 2$ | $-12$ |

Ми йдемо **від помилки назад до входу**, і на кожному рядку використовуємо число з попереднього рядка. Нічого не рахуємо двічі.

У мережі на сто шарів це дуже важливо. Число «як шар 99 впливає на помилку» потрібне для всіх ваг шару 99, шару 98, 97... Порахували один раз, передали назад, помножили на місцевий множник — і маємо відповідь для наступного шару.

**Завдання 2.** Запустіть клітинку: вона робить прямий прохід, потім зворотний, рядок за рядком, як у таблиці.

In [ ]:
# Рахуємо: ПРЯМИЙ ПРОХІД (від входу до помилки)
x, y = 2, 4
w1, w2 = 0.5, 3

h = w1 * x
prediction = w2 * h
error = (prediction - y) ** 2
print("h =", h, "  прогноз =", prediction, "  помилка =", error)

# ЗВОРОТНИЙ ПРОХІД (від помилки назад)
d_prediction = 2 * (prediction - y)    # як прогноз впливає на помилку
d_w2 = d_prediction * h                # крутизна по w2
d_h = d_prediction * w2                # як h впливає на помилку
d_w1 = d_h * x                         # крутизна по w1
print("крутизна по w2 =", d_w2, "  крутизна по w1 =", d_w1)

**Завдання 3.** Зробіть крок градієнтного спуску для **обох** ваг з $\text{lr} = 0.01$ (правило те саме, окремо для кожної ваги) і знову порахуйте помилку. Стало краще?

In [ ]:
# Рахуємо: КРОК для обох ваг і новий прямий прохід
lr = 0.01
w1 = w1 - lr * d_w1
w2 = w2 - lr * d_w2

h = w1 * x
prediction = w2 * h
error = (prediction - y) ** 2
print("w1 =", round(w1, 4), "  w2 =", round(w2, 4), "  помилка =", round(error, 4))

**Ваша відповідь:** _двічі клацніть по цій клітинці, щоб написати_

### Назви

* Правило «вплив через ланцюжок = добуток впливів кожного етапу» (як курси валют) називається **ланцюгове правило** (chain rule). До речі, в кроці 7 ми вже ним користувались, коли множили «$2 \cdot$ помилка» на «$x$».
* Весь алгоритм — прямий прохід, помилка, а потім назад, етап за етапом, ланцюговим правилом — називається **зворотне поширення помилки** (backpropagation, або просто backprop). «Зворотне», бо йдемо від помилки назад. «Поширення», бо число «як це впливає на помилку» передається від шару до шару.
* Список крутизн по всіх вагах (у нас їх дві: $-12$ і $-2$) називається **градієнт**. Звідси й назва «градієнтний спуск»: ідемо проти градієнта, кожну вагу — проти її крутизни.

Історія. Ідею такого зворотного обчислення описав фінський студент Сеппо Ліннайнмаа в магістерській роботі 1970 року. Пол Вербос у 1974 році запропонував застосувати її до нейромереж. Широко відомою вона стала 1986 року після статті Девіда Румельгарта, Джеффрі Гінтона і Рональда Вільямса в журналі *Nature*. З того часу будь-яка нейромережа, від розпізнавання цифр до ChatGPT, навчається саме так: **backprop рахує крутизну для кожної ваги, (mini-batch) SGD робить крок.**

### Проміжний підсумок

* **Що зробили:** Розбили шлях від ваги до помилки на етапи, для кожного етапу порахували, у скільки разів він підсилює зміну, і перемножили (**ланцюгове правило**). Рахували від помилки назад, щоразу використовуючи попередній результат (**backpropagation**). Потім зробили крок для обох ваг тим самим правилом і помилка впала з 1 до 0.065.
* **Навіщо:** Тепер ми вміємо рахувати крутизну для кожної ваги в ланцюжку будь-якої довжини, а значить — навчати нейромережу.
* **Що далі:** У множеннях легко помилитись. Як переконатись, що ми порахували правильно?

---
## Крок 10. Як перевірити, що ми не помилились

Ланцюжки — це множення, а в множеннях легко помилитись: переплутати знак, взяти не той множник.

**Питання 21.** У нас з самого початку є спосіб рахувати крутизну взагалі без ланцюжків. Який? Перевірте ним $-12$ і $-2$.


**Ваша відповідь:** _двічі клацніть по цій клітинці, щоб написати_

In [ ]:
# Рахуємо: перевірка "в лоб" для w1 = 0.5, w2 = 3
x, y = 2, 4
h_small = 0.001

error_start      = (3 * (0.5 * x) - y) ** 2
error_w1_changed = (3 * ((0.5 + h_small) * x) - y) ** 2
error_w2_changed = ((3 + h_small) * (0.5 * x) - y) ** 2

print("крутизна по w1 ≈", round((error_w1_changed - error_start) / h_small, 3), "  (ланцюжок дав -12)")
print("крутизна по w2 ≈", round((error_w2_changed - error_start) / h_small, 3), "  (ланцюжок дав -2)")

Збігається. (Невелика різниця в третьому знаку є тому, що $0.001$ все ж не нескінченно мало, пригадайте таблицю з кроку 4.)

Така перевірка називається **gradient checking**. Нею справді перевіряють програми, які рахують backprop.

**Питання 22.** Якщо «в лоб» так просто, навіщо взагалі backprop? Нехай у мережі мільярд ваг. Скільки прямих проходів знадобиться для одного кроку «в лоб»? А з backprop?


**Ваша відповідь:** _двічі клацніть по цій клітинці, щоб написати_

### Проміжний підсумок

* **Що зробили:** Перевірили результат backprop старим способом із кроку 4: трохи змінили вагу і подивились, як змінилась помилка. Збіглося.
* **Навіщо:** Так перевіряють програми, які рахують backprop. Але для навчання цей спосіб не годиться: на мільярд ваг треба мільярд проходів на кожен крок, а backprop обходиться кількома.
* **Що далі:** Зібрати все разом.

---
## Що ми сьогодні придумали

| Ми сказали своїми словами | Як це називається |
|---|---|
| Перебрати всі варіанти неможливо, шукаємо, дивлячись тільки довкола себе | задача **оптимізації** |
| Наскільки змінюється помилка на одиницю зміни ваги, коли зміна дуже мала | **похідна** |
| $w_{\text{нове}} = w_{\text{старе}} - \text{lr} \cdot \text{крутизна}$ | **градієнтний спуск** |
| Число, на яке множимо крок | **learning rate** |
| Оцінювати крутизну за однією чи кількома випадковими машинами | **SGD / mini-batch** |
| Скільки прикладів беремо на крок | **batch size** |
| Вплив через ланцюжок = добуток впливів, як курси валют | **ланцюгове правило** |
| Ідемо від помилки назад, передаючи вплив від етапу до етапу | **backpropagation** |
| Список крутизн по всіх вагах | **градієнт** |
| Перевірити ланцюжки, трохи змінивши вагу | **gradient checking** |

Весь цикл навчання будь-якої нейромережі:

> взяти кілька випадкових прикладів → прямий прохід → помилка → зворотний прохід (крутизна для кожної ваги) → кожну вагу трохи змінити проти її крутизни → повторити багато разів

---
## Домашнє завдання

Рахувати можна на калькуляторі або в клітинці-калькуляторі внизу.

Ще кілька справжніх машин з того самого датасету:

| Машина | Рік | Маса $x$, т | Витрата $y$, л/100 км |
|---|---|---|---|
| Datsun PL510 | 1970 | 0.97 | 8.7 |
| Fiat 128 | 1973 | 0.85 | 8.1 |
| AMC Hornet | 1970 | 1.26 | 13.1 |
| Pontiac Catalina | 1970 | 2.01 | 16.8 |

**Задача 1. Один крок для однієї машини.** Модель: прогноз $= w \cdot x$. Візьміть Datsun PL510, $w = 5$, $\text{lr} = 0.1$.

1. Порахуйте прогноз і помилку $(\text{прогноз} - y)^2$.
2. Порахуйте крутизну: $2 \cdot (\text{прогноз} - y) \cdot x$.
3. Зробіть крок: $w_{\text{нове}} = w - \text{lr} \cdot \text{крутизна}$.
4. Порахуйте прогноз і помилку з новим $w$. Стало краще?

**Задача 2. Чи згодні машини?** При $w = 5$ і при $w = 9$ порахуйте крутизну окремо для Fiat 128 і для Pontiac Catalina. У якому випадку вони показують в один бік, а в якому — в різні? Поясніть, чому так (підказка: згадайте питання 18).

**Задача 3. Розмір кроку.** Для помилки $(w - 8)^2$ і старту $w = 12$ підберіть (перезапускаючи клітинки з кроку 6) найбільший lr, за якого модель ще не вилітає. Що відбувається рівно на цьому значенні?

**Задача 4. Backprop.** Мережа з кроку 9: $x \to (\times w_1) \to h \to (\times w_2) \to$ прогноз. Нові числа: $x = 1$, $y = 6$, $w_1 = 2$, $w_2 = 2$.

1. Прямий прохід: $h$, прогноз, помилка.
2. Зворотний прохід, заповніть таблицю, як у кроці 9: «як прогноз впливає на помилку», крутизна по $w_2$, «як $h$ впливає на помилку», крутизна по $w_1$.
3. Перевірте обидві крутизни «в лоб», з $0.001$.
4. Зробіть крок з $\text{lr} = 0.05$. Помилка зменшилась?

**Задача 5\* (подумати).** У тій самій мережі на старті $w_1 = 0$ і $w_2 = 0$. Порахуйте обидві крутизни. Що станеться з навчанням? Як ви думаєте, чому ваги нейромереж на старті заповнюють випадковими числами, а не нулями?

In [2]:
# Задача 1

x = 0.97
y = 8.7
w = 5
lr = 0.1

prediction = w * x
error = (prediction - y) ** 2
slope = 2 * (prediction - y) * x

new_w = w - lr * slope
new_prediction = new_w * x
new_error = (new_prediction - y) ** 2

print("Прогноз:", prediction)
print("Помилка:", error)
print("Крутизна:", slope)
print("Нове w:", new_w)
print("Новий прогноз:", new_prediction)
print("Нова помилка:", new_error)


Прогноз: 4.85
Помилка: 14.822499999999998
Крутизна: -7.468999999999999
Нове w: 5.7469
Новий прогноз: 5.574493
Нова помилка: 9.768794007048994


In [3]:

# Задача 2

cars = [
    ("Fiat 128", 0.85, 8.1),
    ("Pontiac Catalina", 2.01, 16.8)
]

for w in [5, 9]:
    print("При w =", w)

    for name, x, y in cars:
        prediction = w * x
        slope = 2 * (prediction - y) * x

        print(name, "Крутизна:", round(slope, 4))

        if slope < 0:
            print("Потрібно збільшити w")
        else:
            print("Потрібно зменшити w")

    print()


При w = 5
Fiat 128 Крутизна: -6.545
Потрібно збільшити w
Pontiac Catalina Крутизна: -27.135
Потрібно збільшити w

При w = 9
Fiat 128 Крутизна: -0.765
Потрібно збільшити w
Pontiac Catalina Крутизна: 5.1858
Потрібно зменшити w



In [4]:

# Задача 3

for lr in [0.5, 0.9, 1.0, 1.1]:
    w = 12

    print("lr =", lr)

    for i in range(5):
        error = (w - 8) ** 2
        print("w =", round(w, 4),
              "Помилка =", round(error, 4))

        slope = 2 * (w - 8)
        w = w - lr * slope

    print()



lr = 0.5
w = 12 Помилка = 16
w = 8.0 Помилка = 0.0
w = 8.0 Помилка = 0.0
w = 8.0 Помилка = 0.0
w = 8.0 Помилка = 0.0

lr = 0.9
w = 12 Помилка = 16
w = 4.8 Помилка = 10.24
w = 10.56 Помилка = 6.5536
w = 5.952 Помилка = 4.1943
w = 9.6384 Помилка = 2.6844

lr = 1.0
w = 12 Помилка = 16
w = 4.0 Помилка = 16.0
w = 12.0 Помилка = 16.0
w = 4.0 Помилка = 16.0
w = 12.0 Помилка = 16.0

lr = 1.1
w = 12 Помилка = 16
w = 3.2 Помилка = 23.04
w = 13.76 Помилка = 33.1776
w = 1.088 Помилка = 47.7757
w = 16.2944 Помилка = 68.7971



In [5]:

# Задача 4. Backprop

x = 1
y = 6
w1 = 2
w2 = 2
lr = 0.05

# 1. Прямий прохід
h = x * w1
prediction = h * w2
error = (prediction - y) ** 2

print("Проміжне значення h:", h)
print("Прогноз:", prediction)
print("Стара помилка:", error)

# 2. Зворотний прохід
d_prediction = 2 * (prediction - y)
gradient_w2 = d_prediction * h
d_h = d_prediction * w2
gradient_w1 = d_h * x

print("Вплив прогнозу на помилку:", d_prediction)
print("Крутизна по w2:", gradient_w2)
print("Вплив h на помилку:", d_h)
print("Крутизна по w1:", gradient_w1)

# 3. Перевірка крутизни
step = 0.001

error_w1 = (x * (w1 + step) * w2 - y) ** 2
check_w1 = (error_w1 - error) / step

error_w2 = (x * w1 * (w2 + step) - y) ** 2
check_w2 = (error_w2 - error) / step

print("Перевірка w1:", round(check_w1, 4))
print("Перевірка w2:", round(check_w2, 4))

# 4. Один крок навчання
w1 = w1 - lr * gradient_w1
w2 = w2 - lr * gradient_w2

new_h = x * w1
new_prediction = new_h * w2
new_error = (new_prediction - y) ** 2

print("Нове w1:", w1)
print("Нове w2:", w2)
print("Новий прогноз:", new_prediction)
print("Нова помилка:", round(new_error, 4))


Проміжне значення h: 2
Прогноз: 4
Стара помилка: 4
Вплив прогнозу на помилку: -4
Крутизна по w2: -8
Вплив h на помилку: -8
Крутизна по w1: -8
Перевірка w1: -7.996
Перевірка w2: -7.996
Нове w1: 2.4
Нове w2: 2.4
Новий прогноз: 5.76
Нова помилка: 0.0576


In [6]:

# Задача 5
x, y = 1, 6
w1, w2 = 0, 0

h = x * w1
prediction = h * w2
error = (prediction - y) ** 2

gradient_w2 = 2 * (prediction - y) * h
gradient_w1 = 2 * (prediction - y) * w2 * x

print("Помилка:", error)
print("Крутизна w1:", gradient_w1)
print("Крутизна w2:", gradient_w2)


Помилка: 36
Крутизна w1: 0
Крутизна w2: 0


Задача 1
Після зміни ваги прогноз став точнішим. Помилка зменшилася з 14.82 до 9.77. Отже, стало краще.

Задача 2
При w = 5 обидві машини показують, що вагу треба збільшити. При w = 9 Fiat показує збільшити, а Pontiac — зменшити. Так відбувається, бо машини мають різну масу та витрату бензину.

Задача 3
Найбільша межа lr — 1. При цьому значенні вага постійно змінюється між 12 і 4, а помилка не зменшується. Якщо lr більше 1, помилка починає зростати.

Після одного кроку навчання помилка зменшилася з 4 до 0.0576. Отже, модель стала робити точніший прогноз.

Задача 5
Обидві крутизни дорівнюють нулю, тому ваги не змінюються і модель не навчається. Саме тому на початку краще використовувати випадкові ваги, а не нулі.

In [ ]:
# Калькулятор: пишіть свої обчислення тут
x = 1
y = 6
w1 = 2
w2 = 2

h = w1 * x
prediction = w2 * h
error = (prediction - y) ** 2
print("h =", h, "  прогноз =", prediction, "  помилка =", error)